# 02. 역전파 (Backpropagation)

## 학습 목표
- Computational Graph로 수식을 표현하고 gradient 흐름을 이해
- Chain Rule을 활용한 역전파 과정을 단계별로 추적
- micrograd 스타일의 자동 미분 엔진 직접 구현
- PyTorch autograd와 결과 비교

## 참고 자료
- [Andrej Karpathy - micrograd](https://www.youtube.com/watch?v=VMj-3S1tku0)
- [CS231n - Backpropagation](https://cs231n.github.io/optimization-2/)

---

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

## 1. Computational Graph

수식을 **연산 노드의 그래프**로 표현하면, 각 노드에서의 local gradient를 구한 뒤 chain rule로 연결하여 전체 gradient를 계산할 수 있다.

### 예시: $f(x, y, z) = (x + y) \cdot z$

```
x = -2 ──┐
          (+) = q = 3 ──┐
y =  5 ──┘              (*) = f = -12
z = -4 ─────────────────┘
```

**Forward**: 왼쪽 -> 오른쪽으로 값을 계산

**Backward**: 오른쪽 -> 왼쪽으로 gradient를 전파

$$\frac{\partial f}{\partial x} = \frac{\partial f}{\partial q} \cdot \frac{\partial q}{\partial x} = z \cdot 1 = -4$$
$$\frac{\partial f}{\partial y} = \frac{\partial f}{\partial q} \cdot \frac{\partial q}{\partial y} = z \cdot 1 = -4$$
$$\frac{\partial f}{\partial z} = q = 3$$

In [ ]:
# Computational Graph 예시: f(x, y, z) = (x + y) * z

# Forward pass
x, y, z = -2.0, 5.0, -4.0
q = x + y          # q = 3
f = q * z           # f = -12
print(f"Forward: x={x}, y={y}, z={z}")
print(f"  q = x + y = {q}")
print(f"  f = q * z = {f}")

# Backward pass (수동 계산)
df_df = 1.0         # 자기 자신에 대한 gradient는 항상 1

# f = q * z에서의 local gradient
df_dq = z           # d(q*z)/dq = z
df_dz = q           # d(q*z)/dz = q

# q = x + y에서의 local gradient
dq_dx = 1.0         # d(x+y)/dx = 1
dq_dy = 1.0         # d(x+y)/dy = 1

# Chain rule
df_dx = df_dq * dq_dx  # = z * 1 = -4
df_dy = df_dq * dq_dy  # = z * 1 = -4

print(f"\nBackward (수동):")
print(f"  df/dq = {df_dq}")
print(f"  df/dz = {df_dz}")
print(f"  df/dx = {df_dx}  (= z * 1)")
print(f"  df/dy = {df_dy}  (= z * 1)")

# PyTorch로 검증
x_t = torch.tensor(x, requires_grad=True)
y_t = torch.tensor(y, requires_grad=True)
z_t = torch.tensor(z, requires_grad=True)
f_t = (x_t + y_t) * z_t
f_t.backward()
print(f"\nPyTorch 검증:")
print(f"  df/dx = {x_t.grad.item()}, df/dy = {y_t.grad.item()}, df/dz = {z_t.grad.item()}")

---
## 2. Chain Rule 복습

합성함수의 미분. 역전파의 수학적 기반.

$$\frac{df}{dx} = \frac{df}{dg} \cdot \frac{dg}{dx}$$

### 예시: $f(x) = (3x + 1)^2$

- $g(x) = 3x + 1$이라 하면 $f(g) = g^2$
- $\frac{df}{dg} = 2g = 2(3x+1)$
- $\frac{dg}{dx} = 3$
- $\frac{df}{dx} = 2(3x+1) \cdot 3 = 6(3x+1)$

In [ ]:
# Chain Rule 예시: f(x) = (3x + 1)^2

# 해석적 미분
def f(x):
    return (3*x + 1)**2

def df_dx_analytical(x):
    return 6 * (3*x + 1)

# 수치 미분 (numerical gradient)
def numerical_gradient(fn, x, eps=1e-5):
    return (fn(x + eps) - fn(x - eps)) / (2 * eps)

test_x = 2.0
print(f"f({test_x}) = {f(test_x)}")
print(f"해석적 미분: df/dx = {df_dx_analytical(test_x)}")
print(f"수치적 미분: df/dx = {numerical_gradient(f, test_x):.6f}")

# 시각화
x_range = np.linspace(-3, 3, 200)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(x_range, f(x_range), 'b-', linewidth=2, label='f(x) = (3x+1)^2')
ax.set_title('Function'); ax.legend(); ax.grid(True, alpha=0.3)
ax.set_xlabel('x'); ax.set_ylabel('f(x)')

ax = axes[1]
ax.plot(x_range, df_dx_analytical(x_range), 'r-', linewidth=2, label="f'(x) = 6(3x+1)")
ax.axhline(y=0, color='k', linewidth=0.5)
ax.set_title('Derivative (Chain Rule)'); ax.legend(); ax.grid(True, alpha=0.3)
ax.set_xlabel('x'); ax.set_ylabel("f'(x)")

plt.tight_layout()
plt.show()

---
## 3. Forward Pass vs Backward Pass

더 복잡한 예시로 단계별로 추적해보자.

### 뉴런 하나의 계산

$$f(x_1, x_2; w_1, w_2, b) = \sigma(w_1 x_1 + w_2 x_2 + b)$$

여기서 $\sigma$는 sigmoid 함수.

```
x1=2 ── (*w1) ──┐
                  (+) ── (+b) ── (sigmoid) ── output
x2=0 ── (*w2) ──┘
```

In [ ]:
# 뉴런 하나의 Forward / Backward 단계별 추적

# 입력과 가중치
x1, x2 = 2.0, 0.0
w1, w2 = -3.0, 4.0
b = -3.0

# ===== Forward Pass =====
print("===== Forward Pass =====")
s1 = w1 * x1           # step 1
print(f"1) w1*x1 = {w1}*{x1} = {s1}")
s2 = w2 * x2           # step 2
print(f"2) w2*x2 = {w2}*{x2} = {s2}")
s3 = s1 + s2           # step 3
print(f"3) s1+s2 = {s1}+{s2} = {s3}")
s4 = s3 + b            # step 4
print(f"4) s3+b  = {s3}+{b} = {s4}")
out = 1 / (1 + np.exp(-s4))  # step 5: sigmoid
print(f"5) sigmoid({s4}) = {out:.6f}")

# ===== Backward Pass =====
print("\n===== Backward Pass =====")
# 최종 출력에 대한 gradient = 1
dout = 1.0
print(f"dL/dout = {dout}")

# sigmoid의 도함수: sigma * (1 - sigma)
ds4 = out * (1 - out) * dout
print(f"dL/ds4 = {out:.6f} * (1-{out:.6f}) * {dout} = {ds4:.6f}")

# s4 = s3 + b
ds3 = 1.0 * ds4
db = 1.0 * ds4
print(f"dL/ds3 = {ds3:.6f}")
print(f"dL/db  = {db:.6f}")

# s3 = s1 + s2
ds1 = 1.0 * ds3
ds2 = 1.0 * ds3

# s1 = w1 * x1
dw1 = x1 * ds1
dx1 = w1 * ds1
print(f"dL/dw1 = x1 * ds1 = {x1} * {ds1:.6f} = {dw1:.6f}")

# s2 = w2 * x2
dw2 = x2 * ds2
dx2 = w2 * ds2
print(f"dL/dw2 = x2 * ds2 = {x2} * {ds2:.6f} = {dw2:.6f}")

# PyTorch 검증
print("\n===== PyTorch 검증 =====")
x1_t = torch.tensor(x1, requires_grad=True)
x2_t = torch.tensor(x2, requires_grad=True)
w1_t = torch.tensor(w1, requires_grad=True)
w2_t = torch.tensor(w2, requires_grad=True)
b_t = torch.tensor(b, requires_grad=True)

out_t = torch.sigmoid(w1_t * x1_t + w2_t * x2_t + b_t)
out_t.backward()

print(f"dL/dw1 = {w1_t.grad.item():.6f} (수동: {dw1:.6f})")
print(f"dL/dw2 = {w2_t.grad.item():.6f} (수동: {dw2:.6f})")
print(f"dL/db  = {b_t.grad.item():.6f} (수동: {db:.6f})")

---
## 4. micrograd 구현

Andrej Karpathy의 [micrograd](https://github.com/karpathy/micrograd)에서 영감을 받은 미니 자동 미분 엔진.

### Value 클래스의 핵심 구조
- `data`: 실제 값
- `grad`: 이 노드에 대한 gradient (backward 후 채워짐)
- `_backward`: 이 노드의 local backward 함수
- `_children`: 이 노드를 만든 입력 노드들

In [ ]:
class Value:
    """micrograd 스타일 자동 미분 엔진"""
    
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None  # default: 아무것도 안 함
        self._children = set(_children)
        self._op = _op
        self.label = label
    
    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        
        def _backward():
            self.grad += 1.0 * out.grad   # d(a+b)/da = 1
            other.grad += 1.0 * out.grad  # d(a+b)/db = 1
        out._backward = _backward
        return out
    
    def __radd__(self, other):
        return self.__add__(other)
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        
        def _backward():
            self.grad += other.data * out.grad   # d(a*b)/da = b
            other.grad += self.data * out.grad   # d(a*b)/db = a
        out._backward = _backward
        return out
    
    def __rmul__(self, other):
        return self.__mul__(other)
    
    def __neg__(self):
        return self * -1
    
    def __sub__(self, other):
        return self + (-other)
    
    def __pow__(self, n):
        assert isinstance(n, (int, float))
        out = Value(self.data ** n, (self,), f'**{n}')
        
        def _backward():
            self.grad += n * (self.data ** (n - 1)) * out.grad
        out._backward = _backward
        return out
    
    def exp(self):
        x = self.data
        out = Value(np.exp(x), (self,), 'exp')
        
        def _backward():
            self.grad += out.data * out.grad  # d(e^x)/dx = e^x
        out._backward = _backward
        return out
    
    def tanh(self):
        x = self.data
        t = np.tanh(x)
        out = Value(t, (self,), 'tanh')
        
        def _backward():
            self.grad += (1 - t**2) * out.grad  # d(tanh)/dx = 1 - tanh^2
        out._backward = _backward
        return out
    
    def sigmoid(self):
        x = self.data
        s = 1 / (1 + np.exp(-x))
        out = Value(s, (self,), 'sigmoid')
        
        def _backward():
            self.grad += s * (1 - s) * out.grad
        out._backward = _backward
        return out
    
    def relu(self):
        out = Value(max(0, self.data), (self,), 'relu')
        
        def _backward():
            self.grad += (1.0 if self.data > 0 else 0.0) * out.grad
        out._backward = _backward
        return out
    
    def backward(self):
        """역전파: topological sort 후 역순으로 _backward 호출"""
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        
        self.grad = 1.0  # 출발점
        for node in reversed(topo):
            node._backward()

print("Value 클래스 구현 완료!")

In [ ]:
# micrograd 테스트: 간단한 수식
# f = (a * b) + c

a = Value(2.0, label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')

d = a * b        # d = -6
e = d + c        # e = 4
e.backward()

print("f = a*b + c")
print(f"a = {a}  (df/da = b = -3)")
print(f"b = {b}  (df/db = a = 2)")
print(f"c = {c}  (df/dc = 1)")
print(f"결과: e = {e.data}")

In [ ]:
# micrograd로 뉴런 계산
# output = tanh(x1*w1 + x2*w2 + b)

x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
b = Value(6.8813735870195432, label='b')  # 특정 값으로 설정

# Forward
x1w1 = x1 * w1
x2w2 = x2 * w2
x1w1x2w2 = x1w1 + x2w2
n = x1w1x2w2 + b
o = n.tanh()

# Backward
o.backward()

print("뉴런: output = tanh(x1*w1 + x2*w2 + b)")
print(f"\nForward:")
print(f"  x1*w1 = {x1w1.data:.4f}")
print(f"  x2*w2 = {x2w2.data:.4f}")
print(f"  sum   = {x1w1x2w2.data:.4f}")
print(f"  +b    = {n.data:.4f}")
print(f"  tanh  = {o.data:.4f}")
print(f"\nBackward (gradients):")
print(f"  do/dw1 = {w1.grad:.4f}")
print(f"  do/dw2 = {w2.grad:.4f}")
print(f"  do/db  = {b.grad:.4f}")
print(f"  do/dx1 = {x1.grad:.4f}")
print(f"  do/dx2 = {x2.grad:.4f}")

---
## 5. micrograd로 뉴런 학습

micrograd의 Value 클래스를 사용하여 간단한 뉴런을 학습시킨다.

목표: 입력 x에 대해 $y = 2x + 1$을 학습

In [ ]:
# micrograd로 간단한 뉴런 학습: y = 2x + 1

# 학습 데이터
X_data = [1.0, 2.0, 3.0, 4.0]
y_data = [3.0, 5.0, 7.0, 9.0]  # y = 2x + 1

# 학습 가능한 파라미터
w = Value(np.random.randn(), label='w')
b_param = Value(np.random.randn(), label='b')

print(f"초기 파라미터: w={w.data:.4f}, b={b_param.data:.4f}")
print(f"목표: w=2.0, b=1.0\n")

lr = 0.01
losses_micrograd = []

for epoch in range(100):
    # 매 epoch마다 gradient 초기화
    w.grad = 0.0
    b_param.grad = 0.0
    
    # 전체 데이터에 대한 loss 계산
    total_loss = Value(0.0)
    for x_val, y_val in zip(X_data, y_data):
        x = Value(x_val)
        y_pred = w * x + b_param      # forward: y = wx + b
        diff = y_pred - Value(y_val)
        loss = diff ** 2              # MSE (하나의 샘플)
        total_loss = total_loss + loss
    
    # Backward
    total_loss.backward()
    losses_micrograd.append(total_loss.data)
    
    # Gradient descent
    w.data -= lr * w.grad
    b_param.data -= lr * b_param.grad
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {total_loss.data:.6f} | w={w.data:.4f}, b={b_param.data:.4f}")

print(f"\n최종: w={w.data:.4f} (목표: 2.0), b={b_param.data:.4f} (목표: 1.0)")

In [ ]:
# 학습 과정 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
ax = axes[0]
ax.plot(losses_micrograd)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('micrograd Training Loss')
ax.grid(True, alpha=0.3)

# 학습 결과
ax = axes[1]
x_plot = np.linspace(0, 5, 100)
y_true = 2 * x_plot + 1
y_learned = w.data * x_plot + b_param.data
ax.plot(x_plot, y_true, 'b-', linewidth=2, label='y = 2x + 1 (target)')
ax.plot(x_plot, y_learned, 'r--', linewidth=2, label=f'y = {w.data:.2f}x + {b_param.data:.2f} (learned)')
ax.scatter(X_data, y_data, c='blue', s=100, zorder=5, label='Training data')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Learned Function')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### micrograd로 XOR 학습

Value 클래스를 사용하여 2-layer 뉴런으로 XOR을 학습한다.

In [ ]:
# micrograd로 XOR 학습 (2-layer network)
np.random.seed(42)

# XOR 데이터
X_xor = [[0, 0], [0, 1], [1, 0], [1, 1]]
y_xor = [0, 1, 1, 0]

# 네트워크 파라미터: 2 input -> 4 hidden (tanh) -> 1 output (sigmoid)
# Hidden layer weights (2x4)
w_h = [[Value(np.random.randn() * 0.5) for _ in range(4)] for _ in range(2)]
b_h = [Value(0.0) for _ in range(4)]
# Output layer weights (4x1)
w_o = [Value(np.random.randn() * 0.5) for _ in range(4)]
b_o = Value(0.0)

all_params = [p for row in w_h for p in row] + b_h + w_o + [b_o]

lr = 1.0
losses_xor_mg = []

for epoch in range(500):
    # Zero gradients
    for p in all_params:
        p.grad = 0.0
    
    total_loss = Value(0.0)
    
    for x_val, y_val in zip(X_xor, y_xor):
        # Hidden layer
        hidden = []
        for j in range(4):
            h = b_h[j]
            for i in range(2):
                h = h + Value(x_val[i]) * w_h[i][j]
            hidden.append(h.tanh())
        
        # Output layer
        out = b_o
        for j in range(4):
            out = out + hidden[j] * w_o[j]
        out = out.sigmoid()
        
        # MSE loss
        diff = out - Value(y_val)
        total_loss = total_loss + diff ** 2
    
    total_loss.backward()
    losses_xor_mg.append(total_loss.data)
    
    # Update
    for p in all_params:
        p.data -= lr * p.grad
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {total_loss.data:.6f}")

# 최종 결과
print("\n=== micrograd XOR 결과 ===")
for x_val, y_val in zip(X_xor, y_xor):
    hidden = []
    for j in range(4):
        h = b_h[j]
        for i in range(2):
            h = h + Value(x_val[i]) * w_h[i][j]
        hidden.append(h.tanh())
    out = b_o
    for j in range(4):
        out = out + hidden[j] * w_o[j]
    out = out.sigmoid()
    print(f"  ({x_val[0]}, {x_val[1]}) -> {out.data:.4f} -> {int(out.data > 0.5)} (target: {y_val})")

---
## 6. PyTorch autograd와 결과 비교

micrograd의 결과가 PyTorch autograd와 정확히 일치하는지 검증한다.

In [ ]:
# 비교 테스트 1: 간단한 수식 f = (a + b) * c
print("=== 테스트 1: f = (a + b) * c ===")

# micrograd
a_mg = Value(2.0)
b_mg = Value(3.0)
c_mg = Value(-4.0)
f_mg = (a_mg + b_mg) * c_mg
f_mg.backward()

# PyTorch
a_pt = torch.tensor(2.0, requires_grad=True)
b_pt = torch.tensor(3.0, requires_grad=True)
c_pt = torch.tensor(-4.0, requires_grad=True)
f_pt = (a_pt + b_pt) * c_pt
f_pt.backward()

print(f"{'':>15} {'micrograd':>12} {'PyTorch':>12} {'Match':>8}")
print(f"{'f':>15} {f_mg.data:>12.4f} {f_pt.item():>12.4f} {f_mg.data == f_pt.item():>8}")
print(f"{'df/da':>15} {a_mg.grad:>12.4f} {a_pt.grad.item():>12.4f} {a_mg.grad == a_pt.grad.item():>8}")
print(f"{'df/db':>15} {b_mg.grad:>12.4f} {b_pt.grad.item():>12.4f} {b_mg.grad == b_pt.grad.item():>8}")
print(f"{'df/dc':>15} {c_mg.grad:>12.4f} {c_pt.grad.item():>12.4f} {c_mg.grad == c_pt.grad.item():>8}")

In [ ]:
# 비교 테스트 2: 뉴런 계산
print("=== 테스트 2: output = tanh(x1*w1 + x2*w2 + b) ===")

# 동일한 값 사용
vals = {'x1': 2.0, 'x2': 3.0, 'w1': -1.0, 'w2': 0.5, 'b': 1.0}

# micrograd
mg = {k: Value(v, label=k) for k, v in vals.items()}
o_mg = (mg['x1'] * mg['w1'] + mg['x2'] * mg['w2'] + mg['b']).tanh()
o_mg.backward()

# PyTorch
pt = {k: torch.tensor(v, requires_grad=True) for k, v in vals.items()}
o_pt = torch.tanh(pt['x1'] * pt['w1'] + pt['x2'] * pt['w2'] + pt['b'])
o_pt.backward()

print(f"\n{'':>15} {'micrograd':>12} {'PyTorch':>12} {'Diff':>12}")
print(f"{'output':>15} {o_mg.data:>12.6f} {o_pt.item():>12.6f} {abs(o_mg.data - o_pt.item()):>12.2e}")
for k in vals:
    mg_grad = mg[k].grad
    pt_grad = pt[k].grad.item()
    print(f"{'d/d'+k:>15} {mg_grad:>12.6f} {pt_grad:>12.6f} {abs(mg_grad - pt_grad):>12.2e}")

In [ ]:
# 비교 테스트 3: 더 복잡한 수식 f = sigmoid(exp(a) + b**2)
print("=== 테스트 3: f = sigmoid(exp(a) + b^2) ===")

# micrograd
a_mg = Value(0.5)
b_mg = Value(-1.5)
f_mg = (a_mg.exp() + b_mg ** 2).sigmoid()
f_mg.backward()

# PyTorch
a_pt = torch.tensor(0.5, requires_grad=True)
b_pt = torch.tensor(-1.5, requires_grad=True)
f_pt = torch.sigmoid(torch.exp(a_pt) + b_pt ** 2)
f_pt.backward()

print(f"{'':>15} {'micrograd':>12} {'PyTorch':>12} {'Diff':>12}")
print(f"{'f':>15} {f_mg.data:>12.6f} {f_pt.item():>12.6f} {abs(f_mg.data - f_pt.item()):>12.2e}")
print(f"{'df/da':>15} {a_mg.grad:>12.6f} {a_pt.grad.item():>12.6f} {abs(a_mg.grad - a_pt.grad.item()):>12.2e}")
print(f"{'df/db':>15} {b_mg.grad:>12.6f} {b_pt.grad.item():>12.6f} {abs(b_mg.grad - b_pt.grad.item()):>12.2e}")

print("\n모든 테스트에서 micrograd와 PyTorch의 결과가 일치합니다!")

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: micrograd에 나눗셈 연산 추가

Value 클래스에 `__truediv__` 메서드를 추가하여 나눗셈을 지원하게 만드세요.

힌트: $a / b = a \cdot b^{-1}$

In [ ]:
# TODO: Value 클래스에 __truediv__ 메서드를 추가하세요
# 힌트: self / other = self * (other ** -1) 로 구현 가능
#
# 테스트:
# a = Value(6.0)
# b = Value(3.0)
# c = a / b
# c.backward()
# print(f"c = {c.data}")  # 2.0
# print(f"dc/da = {a.grad}")  # 1/3 = 0.333...
# print(f"dc/db = {b.grad}")  # -6/9 = -0.666...


### 연습 2: Computational Graph 손으로 계산

아래 수식의 forward/backward를 수동으로 계산한 뒤, micrograd와 PyTorch로 검증하세요.

$$f(a, b) = \tanh(a^2 + 3b)$$

$a = 1.0, \; b = 2.0$일 때:
1. Forward pass: $a^2 = ?$, $3b = ?$, $\text{sum} = ?$, $\tanh(\text{sum}) = ?$
2. Backward pass: $\partial f / \partial a = ?$, $\partial f / \partial b = ?$

In [ ]:
# TODO: f(a, b) = tanh(a^2 + 3b) 의 gradient를 수동 계산하고 검증하세요
# a = 1.0, b = 2.0
#
# 수동 계산:
# Forward: a^2 = 1, 3b = 6, sum = 7, tanh(7) = ?
# Backward:
#   df/d(sum) = 1 - tanh(7)^2
#   df/da = df/d(sum) * d(sum)/d(a^2) * d(a^2)/da = (1-tanh(7)^2) * 1 * 2a
#   df/db = df/d(sum) * d(sum)/d(3b) * d(3b)/db = (1-tanh(7)^2) * 1 * 3
#
# 힌트: micrograd로 계산한 뒤 PyTorch로 검증


---
## 핵심 정리

| 개념 | 핵심 내용 |
|------|----------|
| Computational Graph | 수식을 연산 노드의 그래프로 표현 |
| Chain Rule | 합성함수 미분, 역전파의 수학적 기반 |
| Forward Pass | 입력에서 출력 방향으로 값 계산 |
| Backward Pass | 출력에서 입력 방향으로 gradient 전파 |
| Local Gradient | 각 연산 노드에서의 편미분 |
| micrograd Value | data, grad, _backward, _children으로 자동 미분 |
| Topological Sort | backward 순서를 결정 (의존성 순서의 역순) |
| PyTorch autograd | 동일한 원리를 GPU 최적화하여 구현한 실전 도구 |

**다음 노트북**: [03-cnn-basics.ipynb](03-cnn-basics.ipynb) - CNN과 이미지 분류